In [18]:
import regex as re
from collections import Counter

In [ ]:
text = """u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.
Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."
Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."
They went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could hear them. They were sad and scared, and they never got out of the pit.
<|endoftext|>
"""

In [ ]:
# pretokenize
parts = re.split(
    "|".join(map(re.escape, special_tokens)),
    text
)

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
# we exclude special tokens
special_tokens = ["<|endoftext|>"]

tokens = [] # the list can get massive so we avoid it. 
counts = Counter()

for part in parts: 
    for match in re.finditer(PAT, part):
        token = match.group()
        counts[token] +=1

# now we'll apply the bpe algorithm
# counts is essentially: token: count, token: count and so on . 


In [53]:
num_merges = 20

In [ ]:
def merge_pair(sequence, pair, new_id):
    result = []
    i = 0

    while i < len(sequence):
        if i < len(sequence) - 1 and (sequence[i], sequence[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(sequence[i])
            i += 1

    return result

token_sequences = {
    key: list(key.encode("utf-8"))
    for key in counts
}

vocab = {
    i : bytes([i])
    for i in range(256)
}

merges = []

while len(merges) < num_merges: 

    pair_counts = Counter()

    for key, sequence in token_sequences.items(): 
        freq = counts[key]

        for i in range(len(sequence)-1):
            pair = (sequence[i], sequence[i+1])
            pair_counts[pair] += freq

    best_pair, freq = pair_counts.most_common(1)[0]

    new_id = len(vocab)
    vocab[new_id] = (
        vocab[best_pair[0]] + vocab[best_pair[1]]
    )

    merges.append((best_pair, new_id))

    for key in token_sequences: 
        token_sequences[key] = merge_pair(
            token_sequences[key], 
            best_pair, 
            new_id
        )

return vocab, merges


In [26]:
len(counts), type(counts)

(252, collections.Counter)

In [34]:
sample = {}




In [36]:
sample["h"]=1

In [37]:
"h" in sample

True